In [19]:
#! python -m pip install numpy scipy matplotlib
import scipy as scp
import numpy as np
import matplotlib.pyplot as plt

In [20]:
csi_largo = np.random.randn(500, 64) + 1j * np.random.randn(500, 64)

In [ ]:
csi_ex = np.array([
    # Paquete 1 (t = 0)
    [13.2 + 7.2j,  5.4 + 14.0j, -6.2 + 13.6j, -14.1 + 5.0j,
    -13.4 - 6.6j, -4.6 - 14.3j,  7.0 - 13.3j,  14.4 - 4.2j],
    
    # Paquete 2 (t = 1)
    [11.7 + 9.3j,  3.2 + 14.6j, -8.3 + 12.5j, -14.7 + 3.0j,
    -11.8 - 9.2j, -2.5 - 14.8j,  8.8 - 12.1j,  14.8 - 2.3j],
    
    # Paquete 3 (t = 2)
    [ 9.9 + 11.2j,  0.9 + 15.0j, -10.2 + 11.0j, -14.9 + 0.9j,
    -9.9 - 11.2j, -0.3 - 15.0j,  10.4 - 10.8j,  14.9 - 0.3j]
])

fases = []#lpm


In [22]:
for subportadora in csi_largo:
    fases_z =[]
    for z in subportadora:
        I = z.real  # parte real
        Q = z.imag # parte imaginaria 
        fase = np.arctan2(Q, I) 
        fases_z.append(fase)
        
    fases.append(fases_z)
fases_brutas = np.array(fases)
fases_unwrapped = np.unwrap(fases_brutas, axis=0) 
print(fases_unwrapped)

[[ 4.48530111e-02 -2.84303837e+00  1.55377735e-01 ...  2.36198143e+00
   2.61853122e+00 -1.53172542e+00]
 [-2.53079081e+00 -9.75888787e-01  2.21165963e+00 ...  3.80411275e+00
   5.43097413e+00 -3.97325328e-01]
 [-3.33437875e+00  3.82044267e-01  5.06730804e+00 ...  5.11214703e+00
   8.03867574e+00  2.17665805e+00]
 ...
 [ 5.20128408e+01 -1.73974793e+01 -2.56494963e+01 ...  3.82819563e+01
  -1.00943770e+02  9.04153892e+01]
 [ 4.92623360e+01 -1.54657757e+01 -2.73149644e+01 ...  4.08452466e+01
  -9.95896951e+01  9.06039281e+01]
 [ 4.62964957e+01 -1.64983078e+01 -2.75640106e+01 ...  4.00364863e+01
  -9.77175850e+01  9.11938672e+01]]


In [23]:
fs = 100 #asumo que la frecuencia de lo que me mande hardware será 100 paquetes x seg
nyquist = fs / 2 # tiene que ser la mitad de lo que recibe pq si no tosquea x alguna razon. se llama limite de nyquist
frecuencia_baja, frecuencia_alta = 0.1, 0.5 # 0.1 son 6rpm y 0.5 30rpm
low = frecuencia_baja / nyquist
high = frecuencia_alta / nyquist
b, a = scp.signal.butter(N=2, Wn=[low, high], btype='bandpass') #plantilla del filtro
fases_filtradas = scp.signal.filtfilt(b, a, fases_unwrapped, axis=0) #aplico el filtro a mi fase
print(fases_filtradas)


[[ 13.30138556  -5.29905425 -16.19248816 ...   3.3506165    9.58576474
  -13.04471631]
 [ 13.31225558  -5.33619348 -15.96980222 ...   3.23616547   9.70195898
  -13.01616312]
 [ 13.31870133  -5.37362841 -15.74490356 ...   3.11981431   9.81577136
  -12.98455254]
 ...
 [ -0.09159112   0.03757851  -0.05482295 ...  -0.04071449  -0.06623467
   -0.10780317]
 [ -0.07615325   0.03175062  -0.04611199 ...  -0.03343482  -0.05596978
   -0.09010682]
 [ -0.06257247   0.02651881  -0.03832104 ...  -0.027134    -0.04673419
   -0.07442687]]


In [ ]:
varianzas = np.var(fases_filtradas, axis=0) #calcula varianza
mejor_subportadora = np.argmax(varianzas) #agarro la subportadora de + varianza
mejor_señal = fases_filtradas[:, mejor_subportadora]

# 2. FFT con Zero-Padding (n_fft = 10000 para dar resolución fina en Hz)
n_fft = 10000 
fft_valores = np.fft.fft(mejor_señal, n=n_fft)
magnitudes = np.abs(fft_valores)
frecuencias = np.fft.fftfreq(n_fft, d=1/fs)

# 3. Mapear a frecuencias positivas y a RPM
mitad = n_fft // 2
frecuencias_pos = frecuencias[:mitad]
magnitudes_pos = magnitudes[:mitad]
rpm_pos = frecuencias_pos * 60.0

# 4. Máscara booleana para el rango de respiración humana (6 a 30 RPM)
mascara_humana = (rpm_pos >= 6.0) & (rpm_pos <= 30.0)
rpm_validas = rpm_pos[mascara_humana]
magnitudes_validas = magnitudes_pos[mascara_humana]

# 5. Detección de presencia y cálculo de RPM
if len(magnitudes_validas) > 0:
    indice_pico = np.argmax(magnitudes_validas)
    pico_potencia = magnitudes_validas[indice_pico]
    promedio_ruido = np.mean(magnitudes_pos)
    
    # Criterio: el pico debe destacar sobre el ruido de fondo
    if pico_potencia > (3.0 * promedio_ruido):
        rpm_detectadas = rpm_validas[indice_pico]
        print(f"Presencia detectada: Ritmo: {rpm_detectadas:.1f} RPM (Subportadora {mejor_subportadora})")
    else:
        print("Sin presencia humana detectada ")
        #debería poner algo para ver que sea continuo y que se vaya mostrando el cambio constante. así se ve el rpm en cada momento y aparte si fue un ruido en el rango pero que ocurrio una vez lo saco


Presencia detectada: Ritmo: 12.0 RPM (Subportadora 21)
